# Preprocesamiento del corpus para RAG

Este cuaderno prepara el corpus para el pipeline de RAG en `src`. Todas las transformaciones se justifican desde los hallazgos del EDA.

Objetivos:
- Normalizar metadatos clave (tiempo, porciones, ingredientes) para permitir filtros y ranking.
- Reducir ruido textual sin perder legibilidad en el contexto recuperado.
- Conservar trazabilidad: mantener campos originales cuando sea necesario.

## Conclusiones del EDA que guian el preprocesamiento

- Heterogeneidad en `ingredientes` (cantidades, unidades, sinonimos) -> normalizacion y estructura.
- `tiempo_total` y `porciones` llegan como texto -> se parsean a numerico y se estandarizan para filtros.
- Instrucciones largas y procedimentales -> limpieza ligera (espacios, frases de marca) sin romper el orden.
- Inconsistencias en `n_ingredientes` -> recalcular desde la lista real.
- Duplicados y variantes muy cercanas -> eliminar duplicados exactos para evitar respuestas repetidas.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'ChefGPT_Dataset_Random_Sample.json'
OUTPUT_DIR = ROOT / 'data' / 'preprocesado'
OUTPUT_PATH = OUTPUT_DIR / 'ChefGPT_Dataset_Preprocesado.json'

DATA_PATH

## Carga del corpus

Justificacion: necesitamos inspeccionar la estructura real antes de aplicar limpieza y preservar compatibilidad con el pipeline actual.

In [ ]:
with DATA_PATH.open('r', encoding='utf-8') as handle:
    raw_recipes = json.load(handle)

df = pd.DataFrame(raw_recipes)
df.head(3)

## Validacion de esquema y filtrado minimo

Justificacion: el pipeline requiere ciertos campos y las recetas sin contenido util solo introducen ruido en la recuperacion.

In [ ]:
required_cols = [
    'titulo',
    'ingredientes',
    'instrucciones',
    'tiempo_total',
    'porciones',
    'autor',
    'n_ingredientes',
]

for col in required_cols:
    if col not in df.columns:
        df[col] = None

def ensure_list(value: Any) -> List[str]:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    return [str(value)]

df['ingredientes'] = df['ingredientes'].apply(ensure_list)
df['titulo'] = df['titulo'].fillna('')
df['instrucciones'] = df['instrucciones'].fillna('')

has_content = df['instrucciones'].str.strip().astype(bool) | df['ingredientes'].map(len).gt(0)
before = len(df)
df = df.loc[has_content].reset_index(drop=True)

print(f'Filtrados sin contenido: {before - len(df)}')

## Normalizacion de texto general

Justificacion: se homogeneiza el texto (espacios y frases de marca) para mejorar embeddings y evitar ruido, sin alterar el contenido culinario.

In [ ]:
STOPPHRASES = [
    'directo al paladar',
]

_whitespace_re = re.compile(r'\s+')
_space_before_punct = re.compile(r'\s+([,.;:!?])')

def normalize_whitespace(text: str) -> str:
    if text is None:
        return ''
    text = str(text).replace('\n', ' ').replace('\r', ' ')
    text = _whitespace_re.sub(' ', text)
    text = _space_before_punct.sub(r'\1', text)
    return text.strip()

def remove_stopphrases(text: str, phrases: List[str]) -> str:
    cleaned = text
    for phrase in phrases:
        cleaned = re.sub(re.escape(phrase), '', cleaned, flags=re.IGNORECASE)
    return normalize_whitespace(cleaned)

def clean_text(text: str) -> str:
    return remove_stopphrases(normalize_whitespace(text), STOPPHRASES)

for col in ['titulo', 'instrucciones', 'autor', 'tiempo_total', 'porciones']:
    df[col] = df[col].apply(clean_text)

## Normalizacion de tiempo y porciones

Justificacion: convertir a numerico permite filtros y reranking por tiempo/porciones, y estandarizar el texto mejora la consistencia del indice.

In [ ]:
_number_re = re.compile(r'(\d+(?:[.,]\d+)?)')
_range_re = re.compile(r'(\d+(?:[.,]\d+)?)\s*(?:-|a|hasta)\s*(\d+(?:[.,]\d+)?)')

def _to_float(value: str) -> Optional[float]:
    if value is None:
        return None
    value = value.replace(',', '.').strip()
    if not value:
        return None
    if re.match(r'^\d+/\d+$', value):
        num, den = value.split('/')
        return float(num) / float(den)
    try:
        return float(value)
    except ValueError:
        return None

def parse_minutes(value: str) -> Optional[int]:
    if value is None:
        return None
    text = normalize_whitespace(str(value).lower())
    if not text:
        return None

    m = re.match(r'^(\d{1,2})\s*:\s*(\d{2})$', text)
    if m:
        return int(m.group(1)) * 60 + int(m.group(2))

    hours = 0.0
    minutes = 0.0

    for match in re.finditer(r'(\d+(?:[.,]\d+)?)\s*(?:h|hora|horas)', text):
        num = _to_float(match.group(1))
        if num is not None:
            hours += num

    for match in re.finditer(r'(\d+(?:[.,]\d+)?)\s*(?:m|min|mins|minuto|minutos)', text):
        num = _to_float(match.group(1))
        if num is not None:
            minutes += num

    if 'media hora' in text or 'hora y media' in text:
        minutes += 30

    total = hours * 60 + minutes
    if total > 0:
        return int(round(total))

    m = _number_re.search(text)
    if m:
        num = _to_float(m.group(1))
        if num is not None:
            return int(round(num))
    return None

def format_minutes(minutes: Optional[int]) -> Optional[str]:
    if minutes is None:
        return None
    if minutes < 60:
        return f'{minutes} minutos'
    hours = minutes // 60
    rem = minutes % 60
    if rem:
        return f'{hours} h {rem} min'
    return f'{hours} h'

def parse_portions(value: str) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    if value is None:
        return (None, None, None)
    text = normalize_whitespace(str(value).lower())
    if not text:
        return (None, None, None)

    m = _range_re.search(text)
    if m:
        low = _to_float(m.group(1))
        high = _to_float(m.group(2))
        if low is not None and high is not None:
            avg = (low + high) / 2
            return (avg, low, high)

    m = _number_re.search(text)
    if m:
        num = _to_float(m.group(1))
        if num is not None:
            return (num, num, num)

    return (None, None, None)

def format_portions(num: Optional[float], low: Optional[float], high: Optional[float]) -> Optional[str]:
    if num is None:
        return None
    if low is not None and high is not None and low != high:
        return f'{int(low)}-{int(high)}'
    if float(num).is_integer():
        return str(int(num))
    return str(round(num, 2))

df['tiempo_total_raw'] = df['tiempo_total']
df['tiempo_min'] = df['tiempo_total'].apply(parse_minutes)
df['tiempo_total_norm'] = df['tiempo_min'].apply(format_minutes)
df['tiempo_total'] = df['tiempo_total_norm'].where(df['tiempo_total_norm'].notna(), df['tiempo_total_raw'])

df['porciones_raw'] = df['porciones']
porciones_parsed = df['porciones'].apply(parse_portions)
df['porciones_num'] = porciones_parsed.apply(lambda x: x[0])
df['porciones_min'] = porciones_parsed.apply(lambda x: x[1])
df['porciones_max'] = porciones_parsed.apply(lambda x: x[2])
df['porciones_norm'] = df.apply(
    lambda row: format_portions(row['porciones_num'], row['porciones_min'], row['porciones_max']),
    axis=1,
)
df['porciones'] = df['porciones_norm'].where(df['porciones_norm'].notna(), df['porciones_raw'])

df[['tiempo_total', 'tiempo_min', 'porciones', 'porciones_num']].head(5)

## Normalizacion de ingredientes

Justificacion: las unidades y sinonimos generan ruido en la recuperacion. Estructurar cantidad/unidad/ingrediente mejora filtros y reduce sparsity, manteniendo trazabilidad con el texto original.

In [ ]:
UNIT_ALIASES = {
    'g': 'g',
    'gr': 'g',
    'gramo': 'g',
    'gramos': 'g',
    'kg': 'kg',
    'kilogramo': 'kg',
    'kilogramos': 'kg',
    'ml': 'ml',
    'mililitro': 'ml',
    'mililitros': 'ml',
    'l': 'l',
    'litro': 'l',
    'litros': 'l',
    'cda': 'cda',
    'cucharada': 'cda',
    'cucharadas': 'cda',
    'cdta': 'cdta',
    'cucharadita': 'cdta',
    'cucharaditas': 'cdta',
    'taza': 'taza',
    'tazas': 'taza',
    'pizca': 'pizca',
    'pizcas': 'pizca',
    'ud': 'ud',
    'u': 'ud',
    'unidad': 'ud',
    'unidades': 'ud',
    'count': 'ud',
}

QUALIFIERS = ['al gusto', 'a gusto', 'para servir', 'opcional']

def remove_accents(text: str) -> str:
    return ''.join(
        ch
        for ch in unicodedata.normalize('NFD', text)
        if unicodedata.category(ch) != 'Mn'
    )

def normalize_unit(unit: Optional[str]) -> Optional[str]:
    if not unit:
        return None
    unit = normalize_whitespace(unit).lower()
    return UNIT_ALIASES.get(unit, unit)

quantity_prefix = re.compile(
    r'^\s*(?P<qty>\d+(?:[.,]\d+)?(?:\s+\d+/\d+)?|\d+/\d+)\s*(?P<unit>[a-zA-Z%]+)?\s*(?P<rest>.*)$'
)
FRACTION_RE = re.compile(r'^\d+/\d+$')

def parse_quantity(qty_raw: Optional[str]) -> Optional[float]:
    if not qty_raw:
        return None
    qty_raw = normalize_whitespace(qty_raw)
    if not qty_raw:
        return None
    parts = qty_raw.split()
    if len(parts) == 2 and FRACTION_RE.match(parts[1]):
        base = _to_float(parts[0])
        frac = _to_float(parts[1])
        if base is not None and frac is not None:
            return base + frac
    if FRACTION_RE.match(qty_raw):
        return _to_float(qty_raw)
    return _to_float(qty_raw)

SYNONYMS = {
    'aceite de oliva virgen extra': 'aceite de oliva',
    'aceite de oliva virgen': 'aceite de oliva',
    'aceite oliva': 'aceite de oliva',
    'aove': 'aceite de oliva',
}
SYNONYMS = {remove_accents(k): v for k, v in SYNONYMS.items()}

def normalize_ingredient_name(name: str) -> str:
    cleaned = normalize_whitespace(name).lower()
    for q in QUALIFIERS:
        cleaned = cleaned.replace(q, '')
    cleaned = normalize_whitespace(cleaned)
    key = remove_accents(cleaned)
    return SYNONYMS.get(key, cleaned)

def format_quantity(value: Optional[float]) -> Optional[str]:
    if value is None:
        return None
    if float(value).is_integer():
        return str(int(value))
    return str(round(value, 2)).rstrip('0').rstrip('.')

def normalize_ingredient_line(text: str) -> Dict[str, Any]:
    original = normalize_whitespace(str(text))
    qty = None
    unit = None
    rest = original

    m = quantity_prefix.match(original)
    if m:
        qty = parse_quantity(m.group('qty'))
        unit = normalize_unit(m.group('unit'))
        rest = m.group('rest')

    name = normalize_ingredient_name(rest)
    parts: List[str] = []
    qty_str = format_quantity(qty)
    if qty_str:
        parts.append(qty_str)
    if unit:
        parts.append(unit)
    if name:
        parts.append(name)

    normalized_line = ' '.join(parts).strip()
    if not normalized_line:
        normalized_line = original

    return {
        'cantidad': qty,
        'unidad': unit,
        'ingrediente': name,
        'texto_normalizado': normalized_line,
        'texto_original': original,
    }

def normalize_ingredient_list(items: List[str]) -> Dict[str, Any]:
    lines: List[str] = []
    struct: List[Dict[str, Any]] = []
    norm_names: List[str] = []

    for item in items:
        data = normalize_ingredient_line(item)
        lines.append(data['texto_normalizado'])
        struct.append({
            'cantidad': data['cantidad'],
            'unidad': data['unidad'],
            'ingrediente': data['ingrediente'],
        })
        if data['ingrediente']:
            norm_names.append(data['ingrediente'])

    return {'lines': lines, 'struct': struct, 'norm_names': norm_names}

df['ingredientes_raw'] = df['ingredientes']
normalized = df['ingredientes'].apply(normalize_ingredient_list)
df['ingredientes'] = normalized.apply(lambda x: x['lines'])
df['ingredientes_struct'] = normalized.apply(lambda x: x['struct'])
df['ingredientes_norm'] = normalized.apply(lambda x: x['norm_names'])
df['ingredientes_count_real'] = df['ingredientes'].map(len)
df['n_ingredientes_raw'] = df['n_ingredientes']
df['n_ingredientes'] = df['ingredientes_count_real']

df[['ingredientes_raw', 'ingredientes', 'ingredientes_norm']].head(3)

## Deduplicacion y control de calidad final

Justificacion: eliminar duplicados reduce respuestas repetidas y sesgos en el indice.

In [ ]:
def make_dedupe_key(row: pd.Series) -> str:
    title = normalize_whitespace(row['titulo']).lower()
    instr = normalize_whitespace(row['instrucciones']).lower()
    ing = '|'.join(row['ingredientes_norm'])
    return f'{title}||{ing}||{instr}'

before = len(df)
df['dedupe_key'] = df.apply(make_dedupe_key, axis=1)
df = df.drop_duplicates('dedupe_key').drop(columns=['dedupe_key']).reset_index(drop=True)

print(f'Duplicados eliminados: {before - len(df)}')

summary = {
    'recetas': len(df),
    'tiempo_parseado': float(df['tiempo_min'].notna().mean()),
    'porciones_parseadas': float(df['porciones_num'].notna().mean()),
}
summary

## Guardado del dataset preprocesado

Justificacion: se guarda en una carpeta nueva para separar el corpus original del corpus listo para indexacion.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
records = df.to_dict(orient='records')

with OUTPUT_PATH.open('w', encoding='utf-8') as handle:
    json.dump(records, handle, ensure_ascii=False, indent=2)

print(f'Guardado: {OUTPUT_PATH} ({len(records)} recetas)')

## Siguiente paso

Para construir el indice FAISS con el pipeline actual, usa el JSON preprocesado como entrada:

```bash
python -m src.cli_build_index --data data/preprocesado/ChefGPT_Dataset_Preprocesado.json --out_dir artifacts
```